# Notebook 4: ML Forecasting & Anomaly Detection

This notebook tests whether the teleconnection signals identified in Notebook 3 translate into operational forecast skill for compound wind hydro energy droughts.

**Sub question:** Which forecasting paradigm is most appropriate for climate teleconnection prediction: regularised linear methods, tree based ensembles, boosted ensembles, or sequence models?

**Structure:**
1. Feature engineering based on Notebook 3 findings
2. Walk forward cross validation with nested hyperparameter tuning
3. Four model comparison (ElasticNet, Random Forest, XGBoost, LSTM)
4. Feature group comparison (climate only vs local only vs combined)
5. Feature importance analysis
6. Anomaly detection

**Key inputs from Notebook 3:**
- SAM is the primary compound driver (DJF/MAM, lags 1-4)
- IOD is the strongest individual teleconnection (JJA runoff, lags 2-4 and 8-11)
- ONI is hydro-specific (JJA/SON, lags 0-3)
- Season is essential context (SAM reverses sign between DJF and JJA)
- SAM-WHDI teleconnection strength varies on decadal timescales

## Imports and Data Loading

In [3]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Models
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)

with open("../results/tables/feature_decisions.json", "r") as f:
    feature_decisions = json.load(f)

print(f"Data loaded: {df.shape}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
print(
    f"\nPrimary target: {feature_decisions['primary_target']['variable']} "
    f"at {feature_decisions['primary_target']['lead_time']}-month lead"
)
print(
    f"Secondary target: {feature_decisions['secondary_target']['variable']} "
    f"at {feature_decisions['secondary_target']['lead_time']}-month lead"
)

Data loaded: (564, 13)
Period: 1979-01-01 00:00:00 to 2025-12-01 00:00:00

Primary target: whdi_3 at 3-month lead
Secondary target: whdi_6 at 3-month lead


## Feature Engineering

In [4]:
def create_lagged_features(df, feature_decisions):
    """
    Create the full feature matrix based on the teleconnection findings.

    Returns a dataframe with Group A (climate modes), Group B (local variables), and target variables.
    """
    results = pd.DataFrame(index=df.index)

    # Group A: Climate mode features

    # SAM at lags 1-4
    sam_lags = feature_decisions["features"]["sam"]["lags"]
    for lag in sam_lags:
        results[f"sam_lag{lag}"] = df["sam"].shift(lag)

    # ONI at lags 0-3
    oni_lags = feature_decisions["features"]["oni"]["lags"]
    for lag in oni_lags:
        results[f"oni_lag{lag}"] = df["oni"].shift(lag)

    # IOD at lags 2-4 and 8-11
    iod_lags = feature_decisions["features"]["iod"]["lags"]
    for lag in iod_lags:
        results[f"iod_lag{lag}"] = df["iod"].shift(lag)

    # Season one hot encoding
    # SON is dropped to avoid perfect multicollinearity
    results["season_DJF"] = df.index.month.isin([12, 1, 2]).astype(int)
    results["season_MAM"] = df.index.month.isin([3, 4, 5]).astype(int)
    results["season_JJA"] = df.index.month.isin([6, 7, 8]).astype(int)

    # Group B: Local variable features

    # WHDI3 at t, t-1, t-2
    results["whdi3_lag0"] = df["whdi_3"]
    results["whdi3_lag1"] = df["whdi_3"].shift(1)
    results["whdi3_lag2"] = df["whdi_3"].shift(2)

    # Component anomalies at t
    results["wind_std_lag0"] = df["wind_std"]
    results["runoff_std_lag0"] = df["runoff_std"]

    # Snowmelt anomaly at t (standardise by month)
    snowmelt_monthly_mean = df["snowmelt"].groupby(df.index.month).transform("mean")
    snowmelt_monthly_std = df["snowmelt"].groupby(df.index.month).transform("std")
    results["snowmelt_std_lag0"] = (
        df["snowmelt"] - snowmelt_monthly_mean
    ) / snowmelt_monthly_std

    # Target Variables

    lead = feature_decisions["primary_target"]["lead_time"]
    results["target_whdi3"] = df["whdi_3"].shift(-lead)
    results["target_whdi6"] = df["whdi_6"].shift(-lead)

    return results


# Create featueres
df_features = create_lagged_features(df, feature_decisions)

df_features = df_features.dropna()

print(f"Feature matrix: {df_features.shape}")
print(f"Period after lagging: {df_features.index[0]} to {df_features.index[-1]}")
print(f"Usable months: {len(df_features)}")

# Define feature groups
group_a_cols = [
    c for c in df_features.columns if c.startswith(("sam_", "oni_", "iod_", "season_"))
]
group_b_cols = [
    c
    for c in df_features.columns
    if c.startswith(("whdi3_", "wind_", "runoff_", "snowmelt_", "season_"))
]
all_feature_cols = list(set(group_a_cols + group_b_cols))
target_cols = ["target_whdi3", "target_whdi6"]

print(f"\nGroup A (climate modes): {len(group_a_cols)} features")
print(f"  {group_a_cols}")
print(f"\nGroup B (local conditions): {len(group_b_cols)} features")
print(f"  {group_b_cols}")
print(f"\nCombined: {len(all_feature_cols)} features")

Feature matrix: (547, 26)
Period after lagging: 1979-12-01 00:00:00 to 2025-06-01 00:00:00
Usable months: 547

Group A (climate modes): 18 features
  ['sam_lag1', 'sam_lag2', 'sam_lag3', 'sam_lag4', 'oni_lag0', 'oni_lag1', 'oni_lag2', 'oni_lag3', 'iod_lag2', 'iod_lag3', 'iod_lag4', 'iod_lag8', 'iod_lag9', 'iod_lag10', 'iod_lag11', 'season_DJF', 'season_MAM', 'season_JJA']

Group B (local conditions): 9 features
  ['season_DJF', 'season_MAM', 'season_JJA', 'whdi3_lag0', 'whdi3_lag1', 'whdi3_lag2', 'wind_std_lag0', 'runoff_std_lag0', 'snowmelt_std_lag0']

Combined: 24 features


## Walk Forward CV Setup

In [5]:
def create_walk_forward_folds(df_features, min_train_years=22, test_years=3):
    """
    Create expanding window walk forward cross validaiton folds.

    Each fold has an expanding training window and a fixed size test window.
    """
    folds = []

    years = sorted(df_features.index.year.unique())
    start_year = years[0]

    # First test period starts after the end of the training window (min_train_years)
    first_test_year = start_year + min_train_years

    fold_num = 1
    test_start = first_test_year

    while test_start + test_years <= years[-1] + 1:
        train_mask = df_features.index.year < test_start
        test_mask = (df_features.index.year >= test_start) & (
            df_features.index.year < test_start + test_years
        )

        train_idx = df_features.index[train_mask]
        test_idx = df_features.index[test_mask]

        if len(train_idx) > 0 and len(test_idx) > 0:
            folds.append(
                {
                    "fold": fold_num,
                    "train_start": train_idx[0].year,
                    "train_end": train_idx[-1].year,
                    "test_start": test_idx[0].year,
                    "test_end": test_idx[-1].year,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "train_idx": train_idx,
                    "test_idx": test_idx,
                }
            )
            fold_num += 1

        test_start += test_years

    return folds


folds = create_walk_forward_folds(df_features)

print(" " * 20 + "Walk Forward Cross Validation Folds")
print("=" * 75)
for fold in folds:
    print(
        f"  Fold {fold['fold']}: Train {fold['train_start']}-{fold['train_end']} "
        f"({fold['n_train']} months) | "
        f"Test {fold['test_start']}-{fold['test_end']} "
        f"({fold['n_test']} months)"
    )

                    Walk Forward Cross Validation Folds
  Fold 1: Train 1979-2000 (253 months) | Test 2001-2003 (36 months)
  Fold 2: Train 1979-2003 (289 months) | Test 2004-2006 (36 months)
  Fold 3: Train 1979-2006 (325 months) | Test 2007-2009 (36 months)
  Fold 4: Train 1979-2009 (361 months) | Test 2010-2012 (36 months)
  Fold 5: Train 1979-2012 (397 months) | Test 2013-2015 (36 months)
  Fold 6: Train 1979-2015 (433 months) | Test 2016-2018 (36 months)
  Fold 7: Train 1979-2018 (469 months) | Test 2019-2021 (36 months)
  Fold 8: Train 1979-2021 (505 months) | Test 2022-2024 (36 months)


## Hyperparameter Grids

In [7]:
# Define hyperparameter grids for each model
# Focused on regularisation parameters most relevant to small datasets

param_grids = {
    "elasticnet": {
        "alpha": [0.01, 0.1, 1.0, 10.0],
        "l1_ratio": [0.1, 0.5, 0.9],
    },
    "random_forest": {
        "n_estimators": [100, 300],
        "max_depth": [3, 5, 10, None],
        "min_samples_leaf": [5, 10, 20],
    },
    "xgboost": {
        "n_estimators": [100, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.7, 1.0],
        "reg_alpha": [0, 0.1],
    },
}

# LSTM Config
LSTM_EPOCHS = 100
LSTM_BATCH_SIZE = 32
LSTM_PATIENCE = 10

lstm_config = [
    {"hidden_size": 32, "dropout": 0.2, "lr": 0.001, "seq_length": 6},
    {"hidden_size": 32, "dropout": 0.3, "lr": 0.0005, "seq_length": 12},
    {"hidden_size": 64, "dropout": 0.2, "lr": 0.001, "seq_length": 12},
    {"hidden_size": 64, "dropout": 0.3, "lr": 0.001, "seq_length": 6},
]

print("Hyperparameter search spaces")
for model, grid in param_grids.items():
    total = 1
    for v in grid.values():
        total *= len(v)
    print(f"    {model}: {total} combinations")
print(f"    lstm: {len(lstm_config)} predefined combinations")

Hyperparameter search spaces
    elasticnet: 12 combinations
    random_forest: 24 combinations
    xgboost: 48 combinations
    lstm: 4 predefined combinations
